# Guía Completa sobre Tuplas en Python (`tuple`)

---

## 1. Introducción: ¿Qué es una Tupla?

En Python, una **tupla** (`tuple`) es una estructura de datos secuencial integrada (*built-in*) que permite almacenar una colección de elementos. Sus tres características fundamentales son:

1. **Ordenada:** Los elementos mantienen un orden estricto de inserción y cada uno posee un índice numérico asociado (comenzando desde `0`).
2. **Heterogénea:** Puede almacenar objetos de cualquier tipo de dato simultáneamente (enteros, flotantes, cadenas de texto, booleanos, listas, diccionarios, funciones u otras tuplas).
3. **Inmutable:** Una vez creada en memoria, **su estructura y las referencias a sus elementos no pueden ser alteradas** (no se pueden agregar, eliminar ni reasignar elementos en sus posiciones).

### Comparativa Rápida: Tuplas (`tuple`) vs. Listas (`list`)

| Característica | Tupla (`tuple`) | Lista (`list`) |
| :--- | :--- | :--- |
| **Sintaxis literal** | `(1, 2, 3)` o `1, 2, 3` | `[1, 2, 3]` |
| **Mutabilidad** | **Inmutable** (tamaño y referencias fijas) | **Mutable** (dinámica) |
| **Consumo de memoria** | Menor (bloque fijo en memoria) | Mayor (reserva memoria extra para crecer) |
| **Velocidad de creación/iteración** | Más rápida | Ligeramente más lenta |
| **¿Puede ser clave de un `dict`?** | **Sí** (si sus elementos son *hashables*) | **No** (no es *hashable*) |
| **Propósito semántico** | Registros fijos, coordenadas, datos constantes | Colecciones homogéneas que cambian en el tiempo |

---
## 2. Creación y Estructura de las Tuplas

Existen diversas maneras de instanciar una tupla en Python. Es importante prestar especial atención a la **tupla de un solo elemento (*singleton*)**, donde la coma `,` es el verdadero operador sintáctico que construye la tupla, no los paréntesis.

In [1]:
# 1. Creación estándar con paréntesis
tupla_numeros = (10, 20, 30, 40, 50)
print("Tupla numérica:", tupla_numeros, "| Tipo:", type(tupla_numeros))

# 2. Creación sin paréntesis (Empaquetado de tupla / Tuple Packing)
# En realidad, es la coma (,) la que define la tupla; los paréntesis solo agrupan.
tupla_empaquetada = "Python", 3.12, True
print("Tupla empaquetada:", tupla_empaquetada, "| Tipo:", type(tupla_empaquetada))

# 3. Tupla vacía
tupla_vacia_1 = ()
tupla_vacia_2 = tuple()
print("Tuplas vacías:", tupla_vacia_1, tupla_vacia_2, "| Longitud:", len(tupla_vacia_1))

# 4. CASO CRÍTICO: Tupla de un solo elemento (Singleton)
no_es_tupla = (42)       # Python evalúa esto como una expresión aritmética entre paréntesis
si_es_tupla = (42,)      # La coma al final indica a Python que es una tupla de 1 elemento
tambien_es_tupla = 42,   # Incluso sin paréntesis, la coma crea la tupla

print("\n--- Caso de un solo elemento ---")
print("(42)  -> Valor:", no_es_tupla, "| Tipo:", type(no_es_tupla))
print("(42,) -> Valor:", si_es_tupla, "| Tipo:", type(si_es_tupla))
print("42,   -> Valor:", tambien_es_tupla, "| Tipo:", type(tambien_es_tupla))

# 5. Uso del constructor tuple() a partir de otros iterables (listas, strings, range, sets)
desde_lista = tuple([1, 2, 3])
desde_string = tuple("Tupla")
desde_rango = tuple(range(1, 6))

print("\n--- Constructor tuple() ---")
print("Desde lista :", desde_lista)
print("Desde string:", desde_string)
print("Desde range :", desde_rango)

# 6. Tuplas heterogéneas y anidadas
tupla_compleja = (101, "Datos", (3.14, 2.71), [1, 2], {"clave": "valor"})
print("\nTupla heterogénea y anidada:", tupla_compleja)

Tupla numérica: (10, 20, 30, 40, 50) | Tipo: <class 'tuple'>
Tupla empaquetada: ('Python', 3.12, True) | Tipo: <class 'tuple'>
Tuplas vacías: () () | Longitud: 0

--- Caso de un solo elemento ---
(42)  -> Valor: 42 | Tipo: <class 'int'>
(42,) -> Valor: (42,) | Tipo: <class 'tuple'>
42,   -> Valor: (42,) | Tipo: <class 'tuple'>

--- Constructor tuple() ---
Desde lista : (1, 2, 3)
Desde string: ('T', 'u', 'p', 'l', 'a')
Desde range : (1, 2, 3, 4, 5)

Tupla heterogénea y anidada: (101, 'Datos', (3.14, 2.71), [1, 2], {'clave': 'valor'})


---
## 3. ¿Por qué las Tuplas son Objetos Inmutables? (Explicación Técnica y de Memoria)

### 3.1. Qué significa la inmutabilidad en Python
Un objeto **inmutable** es aquel cuyo estado interno no puede ser modificado después de haber sido creado en memoria. En el caso de una tupla:
- **No se puede reasignar** un elemento en un índice existente (`tupla[0] = nuevo_valor` lanza `TypeError`).
- **No se pueden añadir ni eliminar** posiciones (`append`, `extend`, `pop`, `remove` o `del tupla[0]` no existen ni están permitidos).

### 3.2. Arquitectura en Memoria (CPython)
¿Por qué Python diseñó las tuplas como objetos inmutables existiendo ya las listas?
1. **Asignación de memoria contigua y estática (`PyTupleObject` vs `PyListObject`):**
   - Una **lista (`list`)** es un arreglo dinámico. En CPython, el objeto `PyListObject` mantiene su cabecera por un lado y un puntero separado hacia un bloque de memoria con capacidad extra (*over-allocation*) para que operaciones como `.append()` sean rápidas ($O(1)$ amortizado).
   - Una **tupla (`tuple`)**, al tener un tamaño fijo conocido en el momento de su creación, se almacena en un **único bloque contiguo de memoria** (`PyTupleObject`) que contiene tanto la cabecera como el arreglo exacto de punteros (`PyObject*`) a sus elementos. Esto ahorra memoria RAM y evita una indirección adicional de punteros en la CPU.
2. **Optimización del compilador (*Constant Folding* y caché de tuplas):**
   - Como una tupla de literales constantes nunca va a cambiar, el compilador de *bytecode* de Python puede precalcularla en tiempo de compilación y reutilizar bloques de memoria (*free list*) para tuplas pequeñas y vacías.
3. **Seguridad e Integridad de Datos (*Contratos inmutables*):**
   - Al pasar una tupla a una función o módulo externo, tienes la garantía de que ninguna rutina alterará la secuencia original por accidente (*sin efectos secundarios*).
4. **Capacidad de Hash (*Hashability*):**
   - Para que un objeto pueda ser clave de un diccionario (`dict`) o miembro de un conjunto (`set`), su valor *hash* (`hash(obj)`) debe ser invariante durante toda su vida útil. Como las listas pueden mutar, su hash cambiaría y corrompería la tabla hash; las tuplas, al ser inmutables, **sí son *hashables*** (siempre que todos los elementos que contengan también sean inmutables).

In [2]:
import sys
import timeit

# 1. Demostración del error al intentar modificar una tupla
coordenadas = (10.4806, -66.9036)  # Latitud y longitud
print("Coordenadas originales:", coordenadas)

try:
    coordenadas[0] = 11.0000
except TypeError as error:
    print(f"Error capturado al intentar mutar la tupla: {type(error).__name__} -> {error}")

try:
    del coordenadas[0]
except TypeError as error:
    print(f"Error capturado al intentar eliminar un elemento: {type(error).__name__} -> {error}")

# 2. Comparación de consumo de memoria (Bytes) entre Tupla y Lista con los mismos elementos
elementos_tupla = (1, 2, 3, 4, 5, 6, 7, 8, 9, 10)
elementos_lista = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

bytes_tupla = sys.getsizeof(elementos_tupla)
bytes_lista = sys.getsizeof(elementos_lista)

print("\n--- Comparación de Memoria y Rendimiento ---")
print(f"Tamaño en memoria de la Tupla (10 elem): {bytes_tupla} bytes")
print(f"Tamaño en memoria de la Lista (10 elem): {bytes_lista} bytes")
print(f"Ahorro de memoria con tupla: {bytes_lista - bytes_tupla} bytes ({(1 - bytes_tupla/bytes_lista)*100:.1f}% menos)")

# 3. Comparación de tiempo de instanciación
tiempo_tupla = timeit.timeit("(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)", number=1_000_000)
tiempo_lista = timeit.timeit("[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]", number=1_000_000)
print(f"Tiempo de creación (1 millón de veces) -> Tupla: {tiempo_tupla:.4f}s | Lista: {tiempo_lista:.4f}s")

Coordenadas originales: (10.4806, -66.9036)
Error capturado al intentar mutar la tupla: TypeError -> 'tuple' object does not support item assignment
Error capturado al intentar eliminar un elemento: TypeError -> 'tuple' object doesn't support item deletion

--- Comparación de Memoria y Rendimiento ---
Tamaño en memoria de la Tupla (10 elem): 120 bytes
Tamaño en memoria de la Lista (10 elem): 136 bytes
Ahorro de memoria con tupla: 16 bytes (11.8% menos)
Tiempo de creación (1 millón de veces) -> Tupla: 0.0138s | Lista: 0.1480s


### 3.3. La Sutileza de la Inmutabilidad: Contenedor Inmutable vs. Elementos Mutables

Es fundamental comprender que **una tupla almacena referencias (direcciones de memoria) a objetos**, no copias aisladas.
- La inmutabilidad de la tupla garantiza que **el arreglo de referencias que guarda nunca cambiará** (el índice `i` siempre apuntará exactamente al mismo objeto con el mismo `id()`).
- Sin embargo, si el objeto al que apunta esa referencia es **mutable** (por ejemplo, una `list` o un `dict` dentro de la tupla), **el contenido interno de ese objeto mutable sí puede modificarse** sin que cambie su dirección de memoria. A esto se le conoce como *inmutabilidad superficial (shallow immutability)*.

In [3]:
# Tupla que contiene un string (inmutable), un entero (inmutable) y una lista (mutable)
registro_estudiante = ("Carlos", 21, ["Álgebra", "Cálculo I"])

print("Antes de modificar la lista interna:")
print("Tupla:", registro_estudiante)
print("ID de la tupla           :", id(registro_estudiante))
print("ID de la lista (índice 2):", id(registro_estudiante[2]))

# NO podemos cambiar la referencia del índice 2 por otra lista nueva:
try:
    registro_estudiante[2] = ["Física"]
except TypeError as e:
    print(f"\nIntento de reasignar el índice 2 falló: {e}")

# PERO SÍ podemos mutar el objeto lista al que apunta el índice 2:
registro_estudiante[2].append("Programación en Python")

print("\nDespués de hacer .append() en la lista interna:")
print("Tupla:", registro_estudiante)
print("ID de la tupla           :", id(registro_estudiante), "(¡Sigue siendo exactamente la misma tupla!)")
print("ID de la lista (índice 2):", id(registro_estudiante[2]), "(¡La referencia no cambió, sigue apuntando al mismo objeto lista!)")

Antes de modificar la lista interna:
Tupla: ('Carlos', 21, ['Álgebra', 'Cálculo I'])
ID de la tupla           : 139748770428480
ID de la lista (índice 2): 139748768399936

Intento de reasignar el índice 2 falló: 'tuple' object does not support item assignment

Después de hacer .append() en la lista interna:
Tupla: ('Carlos', 21, ['Álgebra', 'Cálculo I', 'Programación en Python'])
ID de la tupla           : 139748770428480 (¡Sigue siendo exactamente la misma tupla!)
ID de la lista (índice 2): 139748768399936 (¡La referencia no cambió, sigue apuntando al mismo objeto lista!)


### 3.4. Tuplas y Función `hash()`: Uso en Diccionarios y Conjuntos

Una tupla es *hashable* **si y solo si todos sus elementos son inmutables (*hashables*)**. Si una tupla contiene una lista o un diccionario en su interior, Python impedirá calcular su hash para proteger la integridad de los diccionarios y conjuntos.

In [4]:
# Tupla puramente inmutable -> Es hashable y puede ser clave de un diccionario
coord_caracas = (10.4806, -66.9036)
coord_merida = (8.5897, -71.1561)

mapa_ciudades = {
    coord_caracas: "Caracas",
    coord_merida: "Mérida"
}
print(f"Hash de {coord_caracas}: {hash(coord_caracas)}")
print("Consulta por tupla en diccionario:", mapa_ciudades[(10.4806, -66.9036)])

# Tupla con un elemento mutable adentro -> NO es hashable
tupla_con_lista = (1, 2, [3, 4])
try:
    hash(tupla_con_lista)
except TypeError as e:
    print(f"\nError al calcular hash de tupla con lista interna: {e}")

Hash de (10.4806, -66.9036): 8789917580544593353
Consulta por tupla en diccionario: Caracas

Error al calcular hash de tupla con lista interna: unhashable type: 'list'


---
## 4. Acceso a Elementos: Indexación y *Slicing* (Rebanado)

Al ser secuencias ordenadas, las tuplas soportan exactamente el mismo sistema de indexación ($O(1)$) y rebanado (*slicing*, $O(k)$) que las listas y cadenas de texto:
- **Índices positivos:** De izquierda a derecha, comenzando en `0` hasta `len(tupla) - 1`.
- **Índices negativos:** De derecha a izquierda, comenzando en `-1` (último elemento) hasta `-len(tupla)` (primer elemento).
- **Slicing `tupla[inicio:fin:paso]`:** Extrae una **nueva sub-tupla** desde `inicio` (inclusivo) hasta `fin` (exclusivo) avanzando según el `paso`.

In [5]:
dias = ("Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo")

# 1. Indexación simple
print("Primer día (índice 0) :", dias[0])
print("Último día (índice -1):", dias[-1])
print("Penúltimo día (-2)    :", dias[-2])

# 2. Slicing (Rebanado) -> Siempre devuelve una nueva tupla
dias_laborales = dias[0:5]     # Del índice 0 al 4 (el 5 no se incluye)
fin_de_semana = dias[5:]       # Desde el índice 5 hasta el final
primeros_tres = dias[:3]       # Desde el inicio hasta el índice 2
dias_alternos = dias[::2]      # Toda la tupla dando saltos de 2 en 2
tupla_invertida = dias[::-1]   # Invierte el orden de la tupla con paso -1

print("\n--- Ejemplos de Slicing ---")
print("Días laborales :", dias_laborales)
print("Fin de semana  :", fin_de_semana)
print("Días alternos  :", dias_alternos)
print("Tupla invertida:", tupla_invertida)

# 3. Indexación en tuplas anidadas (Matrices o estructuras multidimensionales)
matriz_inmutable = (
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 9)
)
print("\n--- Tuplas Anidadas ---")
print("Fila 1 completa           :", matriz_inmutable[1])
print("Elemento en Fila 2, Col 0 :", matriz_inmutable[2][0])  # Valor 7

Primer día (índice 0) : Lunes
Último día (índice -1): Domingo
Penúltimo día (-2)    : Sábado

--- Ejemplos de Slicing ---
Días laborales : ('Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes')
Fin de semana  : ('Sábado', 'Domingo')
Días alternos  : ('Lunes', 'Miércoles', 'Viernes', 'Domingo')
Tupla invertida: ('Domingo', 'Sábado', 'Viernes', 'Jueves', 'Miércoles', 'Martes', 'Lunes')

--- Tuplas Anidadas ---
Fila 1 completa           : (4, 5, 6)
Elemento en Fila 2, Col 0 : 7


---
## 5. Operaciones con Tuplas y "Modificación" Indirecta

Aunque una tupla no se puede mutar *in-place*, podemos aplicar operadores algebraicos y relacionales para **generar nuevas tuplas** o evaluar condiciones.

### 5.1. Concatenación, Repetición, Pertenencia y Comparación

In [6]:
tupla_a = (1, 2, 3)
tupla_b = (4, 5, 6)

# 1. Concatenación (+) -> Crea una NUEVA tupla en memoria
tupla_unida = tupla_a + tupla_b
print("Concatenación (tupla_a + tupla_b):", tupla_unida)

# 2. Repetición (*) -> Repite la secuencia N veces en una nueva tupla
tupla_repetida = ("Eco",) * 4
print("Repetición (('Eco',) * 4)        :", tupla_repetida)

# 3. Operadores de pertenencia (in / not in) -> Búsqueda lineal O(n)
print("¿Está el 3 en tupla_a?  :", 3 in tupla_a)
print("¿No está el 9 en tupla_a?:", 9 not in tupla_a)

# 4. Comparación lexicográfica (elemento por elemento de izquierda a derecha)
print("\n--- Comparación Lexicográfica ---")
print("(1, 2, 3) == (1, 2, 3) :", (1, 2, 3) == (1, 2, 3))
print("(1, 2, 3) <  (1, 2, 10):", (1, 2, 3) < (1, 2, 10), "-> Compara el primer elemento distinto (3 < 10)")
print("(1, 5, 0) >  (1, 2, 99):", (1, 5, 0) > (1, 2, 99), "-> Se decide en el segundo elemento (5 > 2)")

Concatenación (tupla_a + tupla_b): (1, 2, 3, 4, 5, 6)
Repetición (('Eco',) * 4)        : ('Eco', 'Eco', 'Eco', 'Eco')
¿Está el 3 en tupla_a?  : True
¿No está el 9 en tupla_a?: True

--- Comparación Lexicográfica ---
(1, 2, 3) == (1, 2, 3) : True
(1, 2, 3) <  (1, 2, 10): True -> Compara el primer elemento distinto (3 < 10)
(1, 5, 0) >  (1, 2, 99): True -> Se decide en el segundo elemento (5 > 2)


### 5.2. ¿Cómo "agregar", "modificar" o "eliminar" datos si es inmutable?

Cuando necesitamos actualizar una variable que contiene una tupla, no modificamos el objeto original, sino que **construimos una nueva tupla y reasignamos la variable**. Tenemos dos patrones principales:
1. **Reasignación mediante concatenación (`+`, `+=`) y *slicing*.**
2. **Conversión temporal a `list`**, aplicación de cambios y reconversión a `tuple`.

In [7]:
versiones = (1.0, 1.1, 1.2)
id_original = id(versiones)
print(f"Tupla inicial: {versiones} | ID en memoria: {id_original}")

# Método 1: Concatenación con += (Crea una nueva tupla y reasigna la variable)
versiones += (2.0,)
print(f"Tras += (2.0,): {versiones} | Nuevo ID: {id(versiones)} (¿Cambió el ID?: {id(versiones) != id_original})")

# Método 2: Insertar o reemplazar un elemento usando Slicing
# Queremos cambiar el índice 1 (valor 1.1) por 1.15
versiones = versiones[:1] + (1.15,) + versiones[2:]
print(f"Tras reemplazo con slicing: {versiones}")

# Método 3: Conversión temporal a lista (ideal si se harán múltiples modificaciones)
temporal = list(versiones)
temporal.append(3.0)
temporal.remove(1.0)
versiones = tuple(temporal)
print(f"Tras conversión list -> tuple: {versiones}")

Tupla inicial: (1.0, 1.1, 1.2) | ID en memoria: 139748770429952
Tras += (2.0,): (1.0, 1.1, 1.2, 2.0) | Nuevo ID: 139748766689104 (¿Cambió el ID?: True)
Tras reemplazo con slicing: (1.0, 1.15, 1.2, 2.0)
Tras conversión list -> tuple: (1.15, 1.2, 2.0, 3.0)


---
## 6. Empaquetado y Desempaquetado de Tuplas (*Packing & Unpacking*)

Una de las características más elegantes y expresivas de Python es el **desempaquetado de secuencias**. Permite asignar los elementos de una tupla a múltiples variables individuales en una sola instrucción.

- **Desempaquetado estricto:** El número de variables a la izquierda debe coincidir exactamente con la longitud de la tupla (de lo contrario lanza `ValueError`).
- **Desempaquetado extendido (Operador `*`):** Permite capturar múltiples elementos sobrantes en una **lista**.
- **Convención `_`:** Se utiliza el guion bajo `_` para descartar valores que no necesitamos utilizar.

In [8]:
# 1. Empaquetado (Packing)
punto_3d = 15, -8, 42  # Se empaquetan 3 valores en una tupla

# 2. Desempaquetado clásico (Unpacking)
x, y, z = punto_3d
print(f"Coordenadas desempaquetadas -> x={x}, y={y}, z={z}")

# 3. Intercambio de variables (Swap) sin variable temporal
a = "Valor A"
b = "Valor B"
a, b = b, a  # A la derecha se crea la tupla (b, a) y se desempaqueta en a, b
print(f"Intercambio directo -> a='{a}', b='{b}'")

# 4. Desempaquetado extendido con el operador asterisco (*)
calificaciones = (12, 15, 16, 18, 19, 20)
peor_nota, *notas_intermedias, mejor_nota = calificaciones

print("\n--- Desempaquetado con * ---")
print("Peor nota        :", peor_nota)
print("Notas intermedias:", notas_intermedias, "| Nota: ¡se empaquetan como una lista!")
print("Mejor nota       :", mejor_nota)

# 5. Ignorar elementos con el comodín (_)
registro_usuario = ("USR-001", "ana_perez", "hash_secreto_xyz", "admin", "2026-10-14")
id_usuario, username, _, rol, _ = registro_usuario
print(f"\nUsuario extraído: {username} (ID: {id_usuario}, Rol: {rol})")

# 6. Retorno múltiple en funciones (En realidad la función retorna UNA tupla)
def estadisticas_basicas(datos):
    minimo = min(datos)
    maximo = max(datos)
    promedio = sum(datos) / len(datos)
    return minimo, maximo, promedio  # Empaqueta y retorna una tupla de 3 elementos

resultado_tupla = estadisticas_basicas((10, 20, 30, 40, 50))
print("\nTipo devuelto por la función:", type(resultado_tupla), "->", resultado_tupla)

# Desempaquetado directo al llamar a la función:
val_min, val_max, val_prom = estadisticas_basicas((10, 20, 30, 40, 50))
print(f"Mínimo: {val_min} | Máximo: {val_max} | Promedio: {val_prom}")

Coordenadas desempaquetadas -> x=15, y=-8, z=42
Intercambio directo -> a='Valor B', b='Valor A'

--- Desempaquetado con * ---
Peor nota        : 12
Notas intermedias: [15, 16, 18, 19] | Nota: ¡se empaquetan como una lista!
Mejor nota       : 20

Usuario extraído: ana_perez (ID: USR-001, Rol: admin)

Tipo devuelto por la función: <class 'tuple'> -> (10, 50, 30.0)
Mínimo: 10 | Máximo: 50 | Promedio: 30.0


---
## 7. Métodos Nativos de las Tuplas

Debido a que las tuplas son inmutables, **no poseen métodos que modifiquen su contenido** (como `.append()`, `.insert()`, `.remove()`, `.pop()`, `.sort()` o `.reverse()`).

El objeto `tuple` cuenta únicamente con **dos métodos públicos**:

1. **`tupla.count(valor)`**: Devuelve un número entero con la cantidad de veces que `valor` aparece dentro de la tupla. Si no existe, devuelve `0`.
2. **`tupla.index(valor[, inicio[, fin]])`**: Devuelve el índice (base 0) de la **primera aparición** de `valor`. Permite acotar la búsqueda entre los índices opcionales `inicio` y `fin`. Si el elemento no se encuentra, lanza una excepción `ValueError`.

> **Nota sobre booleanos y enteros:** En Python, `bool` es subclase de `int` (`True == 1` y `False == 0`). Tanto `.count()` como `.index()` evalúan igualdad (`==`), por lo que `1` y `True` se consideran iguales en la búsqueda.

In [9]:
# Inspeccionemos los métodos públicos de la clase tuple
metodos_publicos = [m for m in dir(tuple) if not m.startswith("_")]
print("Métodos públicos de 'tuple':", metodos_publicos)

# Tupla de ejemplo
respuestas = ("A", "C", "B", "A", "D", "A", "B", "C", "A")

# 1. Uso de .count(valor)
conteo_A = respuestas.count("A")
conteo_Z = respuestas.count("Z")
print(f"\nLa respuesta 'A' aparece {conteo_A} veces.")
print(f"La respuesta 'Z' aparece {conteo_Z} veces.")

# 2. Uso de .index(valor, inicio, fin)
primer_indice_A = respuestas.index("A")
segundo_indice_A = respuestas.index("A", primer_indice_A + 1)  # Busca a partir del siguiente índice
tercer_indice_A = respuestas.index("A", segundo_indice_A + 1, 7)

print(f"\nPrimera aparición de 'A' en el índice : {primer_indice_A}")
print(f"Segunda aparición de 'A' en el índice : {segundo_indice_A}")
print(f"Tercera aparición de 'A' en el índice : {tercer_indice_A}")

# Manejo seguro cuando un elemento puede no existir
elemento_buscado = "E"
if elemento_buscado in respuestas:
    print(f"Índice de {elemento_buscado}: {respuestas.index(elemento_buscado)}")
else:
    print(f"El elemento '{elemento_buscado}' no se encuentra en la tupla (evitamos ValueError).")

Métodos públicos de 'tuple': ['count', 'index']

La respuesta 'A' aparece 4 veces.
La respuesta 'Z' aparece 0 veces.

Primera aparición de 'A' en el índice : 0
Segunda aparición de 'A' en el índice : 3
Tercera aparición de 'A' en el índice : 5
El elemento 'E' no se encuentra en la tupla (evitamos ValueError).


---
## 8. Funciones Integradas (*Built-in Functions*) Asociadas a las Tuplas

Python ofrece un amplio conjunto de funciones nativas que operan sobre cualquier iterable, incluidas las tuplas:

| Función | Descripción |
| :--- | :--- |
| `len(tupla)` | Retorna la cantidad de elementos de la tupla ($O(1)$). |
| `min(tupla)` / `max(tupla)` | Retorna el elemento menor / mayor (requiere elementos comparables entre sí). |
| `sum(tupla[, inicio])` | Suma todos los elementos numéricos de la tupla. |
| `sorted(tupla, reverse=False, key=None)` | Retorna una **nueva lista (`list`)** con los elementos ordenados. |
| `reversed(tupla)` | Retorna un iterador inverso sobre los elementos de la tupla. |
| `all(tupla)` | Retorna `True` si **todos** los elementos tienen valor booleano verdadero (*truthy*). |
| `any(tupla)` | Retorna `True` si **al menos un** elemento es verdadero (*truthy*). |
| `enumerate(tupla, start=0)` | Genera pares de tuplas `(índice, valor)` durante una iteración. |
| `zip(tupla1, tupla2, ...)` | Combina múltiples iterables emparejando sus elementos en tuplas posición a posición. |

In [10]:
precios = (45.50, 12.00, 89.99, 23.40, 12.00, 60.00)

# 1. Funciones de agregación y longitud
print("Longitud len() :", len(precios))
print("Mínimo min()   :", min(precios))
print("Máximo max()   :", max(precios))
print("Suma sum()     :", round(sum(precios), 2))

# 2. Ordenamiento con sorted()
# IMPORTANTE: sorted() siempre devuelve una LISTA. Si deseamos una tupla, envolvemos con tuple().
lista_ordenada = sorted(precios)
tupla_ordenada_desc = tuple(sorted(precios, reverse=True))
print("\nResultado de sorted() (es lista)      :", lista_ordenada)
print("Convertido de nuevo a tupla (desc)    :", tupla_ordenada_desc)

# 3. Evaluación lógica con all() y any()
validaciones = (True, True, 1, "Texto no vacío")
permisos = (False, 0, "", True)
print("\nall(validaciones) -> ¿Todos son verdaderos? :", all(validaciones))
print("any(permisos)     -> ¿Alguno es verdadero?  :", any(permisos))

# 4. Iteración con enumerate() y emparejamiento con zip()
productos = ("Laptop", "Mouse", "Teclado")
costos = (850, 25, 60)

print("\n--- Uso de enumerate() ---")
for indice, prod in enumerate(productos, start=1):
    print(f"  #{indice}: {prod}")

print("\n--- Uso de zip() ---")
catalogo = tuple(zip(productos, costos))
print("Tupla generada con zip():", catalogo)
for prod, costo in catalogo:
    print(f"  Producto: {prod:<10} | Precio: ${costo}")

Longitud len() : 6
Mínimo min()   : 12.0
Máximo max()   : 89.99
Suma sum()     : 242.89

Resultado de sorted() (es lista)      : [12.0, 12.0, 23.4, 45.5, 60.0, 89.99]
Convertido de nuevo a tupla (desc)    : (89.99, 60.0, 45.5, 23.4, 12.0, 12.0)

all(validaciones) -> ¿Todos son verdaderos? : True
any(permisos)     -> ¿Alguno es verdadero?  : True

--- Uso de enumerate() ---
  #1: Laptop
  #2: Mouse
  #3: Teclado

--- Uso de zip() ---
Tupla generada con zip(): (('Laptop', 850), ('Mouse', 25), ('Teclado', 60))
  Producto: Laptop     | Precio: $850
  Producto: Mouse      | Precio: $25
  Producto: Teclado    | Precio: $60


---
## 9. ¿Existe la "Comprensión de Tuplas"? (Expresiones Generadoras)

Un error muy frecuente al aprender Python es pensar que escribir `(x**2 for x in range(5))` crea una tupla por comprensión (de forma análoga a `[x**2 for x in range(5)]` con las listas).

- Al usar paréntesis `(expresion for item in iterable)`, Python crea una **Expresión Generadora (`generator`)**, que evalúa los elementos de forma perezosa (*lazy evaluation*) uno por uno bajo demanda para ahorrar memoria.
- Para crear efectivamente una **tupla a partir de una comprensión**, debemos pasar la expresión generadora al constructor `tuple()`.

In [11]:
# 1. Paréntesis solos crean un Generador (Generator Expression)
objeto_generador = (x**2 for x in range(1, 6))
print("Con paréntesis solos :", objeto_generador, "| Tipo:", type(objeto_generador))

# 2. Para obtener una tupla, usamos el constructor tuple()
tupla_cuadrados = tuple(x**2 for x in range(1, 6))
print("Con constructor tuple:", tupla_cuadrados, "| Tipo:", type(tupla_cuadrados))

# 3. Ejemplo con filtro condicional
tupla_pares = tuple(n for n in range(20) if n % 2 == 0)
print("Tupla de números pares:", tupla_pares)

Con paréntesis solos : <generator object <genexpr> at 0x7f19cb9a8c10> | Tipo: <class 'generator'>
Con constructor tuple: (1, 4, 9, 16, 25) | Tipo: <class 'tuple'>
Tupla de números pares: (0, 2, 4, 6, 8, 10, 12, 14, 16, 18)


---
## 10. Nivel Avanzado: Tuplas con Nombre (`collections.namedtuple`)

Cuando una tupla tiene muchos campos (por ejemplo, un registro de base de datos con múltiples columnas), recordar qué significa `registro[0]`, `registro[2]` o `registro[3]` hace que el código sea propenso a errores y difícil de leer.

El módulo estándar `collections` incluye `namedtuple`, una fábrica de subclases de `tuple` que permite acceder a los elementos **tanto por su índice numérico como por su nombre de atributo** (`objeto.campo`), **manteniendo intacta la inmutabilidad, la eficiencia y el bajo consumo de memoria de una tupla estándar**.

In [12]:
from collections import namedtuple

# Definimos la estructura de la tupla con nombre 'PuntoGeo'
PuntoGeo = namedtuple("PuntoGeo", ["ciudad", "latitud", "longitud", "altitud_msnm"])

# Instanciamos objetos como si fuera una clase
ubicacion_1 = PuntoGeo("Caracas", 10.4806, -66.9036, 900)
ubicacion_2 = PuntoGeo(ciudad="Maracaibo", latitud=10.6427, longitud=-71.6125, altitud_msnm=6)

print("Representación legible:", ubicacion_1)
print("¿Es una subclase de tuple?:", isinstance(ubicacion_1, tuple))

# Acceso por nombre de atributo (mucho más claro) y por índice tradicional
print(f"\nCiudad: {ubicacion_1.ciudad} | Altitud: {ubicacion_1.altitud_msnm} msnm")
print(f"Acceso por índice [0]: {ubicacion_1[0]}")

# Métodos útiles propios de namedtuple: _asdict() y _replace()
print("\nConversión a diccionario (_asdict):", ubicacion_1._asdict())

# _replace() NO muta la tupla original, sino que retorna una NUEVA instancia con el campo actualizado
ubicacion_1_actualizada = ubicacion_1._replace(altitud_msnm=920)
print("Nueva instancia con _replace()     :", ubicacion_1_actualizada)

Representación legible: PuntoGeo(ciudad='Caracas', latitud=10.4806, longitud=-66.9036, altitud_msnm=900)
¿Es una subclase de tuple?: True

Ciudad: Caracas | Altitud: 900 msnm
Acceso por índice [0]: Caracas

Conversión a diccionario (_asdict): {'ciudad': 'Caracas', 'latitud': 10.4806, 'longitud': -66.9036, 'altitud_msnm': 900}
Nueva instancia con _replace()     : PuntoGeo(ciudad='Caracas', latitud=10.4806, longitud=-66.9036, altitud_msnm=920)


---
## 11. Resumen y Buenas Prácticas: ¿Cuándo usar Tuplas?

1. **Integridad de datos (Inmutabilidad por diseño):** Úsalas cuando una colección de valores no deba cambiar a lo largo de la ejecución del programa (ej. configuraciones, colores RGB `(255, 128, 0)`, coordenadas geográficas, días de la semana).
2. **Registros heterogéneos (estructuras tipo fila/registro):** Si cada posición tiene un significado semántico distinto (ej. `(id_empleado, nombre, salario)`), usa una tupla (o `namedtuple`). Si todos los elementos son del mismo tipo y la colección crece o disminuye dinámicamente (ej. lista de nombres de clientes), usa una `list`.
3. **Claves compuestas en diccionarios:** Cuando necesites indexar un diccionario por más de un criterio, como `(año, mes)` o `(latitud, longitud)`, las tuplas son la estructura ideal por ser *hashables*.
4. **Retorno de múltiples valores en funciones:** Permite devolver múltiples resultados empaquetados de forma limpia y desempaquetarlos directamente en la llamada.
5. **Optimización de memoria:** Si vas a almacenar millones de secuencias estáticas en memoria para solo lectura, usar tuplas en lugar de listas reducirá notablemente el consumo de RAM.